In [1]:
import numpy as np

import os
import tensorflow as tf

from models import ConvNet, SoftBinaryDecisionTree
from models.unlearning import UnlearningSDT
from models.utils import brand_new_tfsession, draw_tree
from tensorflow.keras.callbacks import EarlyStopping, Callback

sess = brand_new_tfsession()

c:\Users\kimdd\anaconda3\envs\unlearning\lib\site-packages\tensorflow\python\framework\dtypes.py:526: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
c:\Users\kimdd\anaconda3\envs\unlearning\lib\site-packages\tensorflow\python\framework\dtypes.py:527: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_quint8 = np.dtype([("quint8", np.uint8, 1)])
c:\Users\kimdd\anaconda3\envs\unlearning\lib\site-packages\tensorflow\python\framework\dtypes.py:528: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint16 = np.dtype([("qint16", np.int16, 1)])
c:\Users\kimdd\anaconda3\envs\unlearning\lib\site-packages

In [2]:
# load MNIST data
mnist = tf.keras.datasets.mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# add channel dim
x_train = x_train[..., np.newaxis]
x_test = x_test[..., np.newaxis]

# ❗️Remove samples with target class 3 from training data only
train_mask = y_train != 3
x_train = x_train[train_mask]
y_train = y_train[train_mask]

# ✅ test set은 원본 그대로 유지 (class 3 포함)

# Split train/valid
x_valid, y_valid = x_train[-10000:], y_train[-10000:]
x_train, y_train = x_train[:-10000], y_train[:-10000]

print(x_train.shape, y_train.shape, x_valid.shape, y_valid.shape, x_test.shape, y_test.shape)

(43869, 28, 28, 1) (43869,) (10000, 28, 28, 1) (10000,) (10000, 28, 28, 1) (10000,)


In [3]:
# retrieve image and label shapes from training data
img_rows, img_cols, img_chans = x_train.shape[1:]
# n_classes = np.unique(y_train).shape[0]

n_classes = 10

print(img_rows, img_cols, img_chans, n_classes)

28 28 1 10


In [4]:
# convert labels to 1-hot vectors
y_train = tf.keras.utils.to_categorical(y_train, n_classes)
y_valid = tf.keras.utils.to_categorical(y_valid, n_classes)
y_test = tf.keras.utils.to_categorical(y_test, n_classes)

print(y_train.shape, y_valid.shape, y_test.shape)

(43869, 10) (10000, 10) (10000, 10)


In [5]:
# normalize inputs and cast to float
x_train = (x_train / np.max(x_train)).astype(np.float32)
x_valid = (x_valid / np.max(x_valid)).astype(np.float32)
x_test = (x_test / np.max(x_test)).astype(np.float32)

In [ ]:
import time
start_time = time.time()
print("🟢 학습 시작:", time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(start_time)))

test_nn = ConvNet(img_rows, img_cols, img_chans, n_classes, model_path='assets/mnist_teacher/test-mnist-model.hdf5')
test_nn.maybe_train(data_train=(x_train, y_train),
               data_valid=(x_valid, y_valid),
               batch_size=16, epochs=20)

# 4. 종료 시간 기록
end_time = time.time()
print("🔴 학습 종료:", time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(end_time)))

# 5. 총 소요 시간
elapsed = end_time - start_time
print(f"⏱️ 총 학습 시간: {elapsed:.2f}초 ({elapsed / 60:.2f}분)")

🟢 학습 시작: 2025-05-29 12:20:47
Instructions for updating:
Colocations handled automatically by placer.
🚀 No checkpoint found on assets/test2-mnist-model.hdf5. Training from scratch.
Instructions for updating:
Please use `rate` instead of `keep_prob`. Rate should be set to `rate = 1 - keep_prob`.
Train on 43869 samples, validate on 10000 samples
Instructions for updating:
Use tf.cast instead.
Epoch 1/20
43869/43869 [==============================] - 61s 1ms/sample - loss: 0.1844 - acc: 0.9397 - val_loss: 0.0497 - val_acc: 0.9869
Epoch 2/20
43869/43869 [==============================] - 60s 1ms/sample - loss: 0.0618 - acc: 0.9808 - val_loss: 0.0423 - val_acc: 0.9889
Epoch 3/20
43869/43869 [==============================] - 60s 1ms/sample - loss: 0.0442 - acc: 0.9861 - val_loss: 0.0365 - val_acc: 0.9905
Epoch 4/20
43869/43869 [==============================] - 60s 1ms/sample - loss: 0.0356 - acc: 0.9880 - val_loss: 0.0458 - val_acc: 0.9881
Epoch 5/20
43869/43869 [===========================

In [46]:
test_nn.evaluate(x_valid, y_valid)
test_nn.evaluate(x_test, y_test)

Evaluation Results:
 - Loss:      0.0348
 - Accuracy:  99.26%
 - Precision: 99.26%
 - Recall:    99.24%
 - F1-Score:  99.25%
Evaluation Results:
 - Loss:      1.6469
 - Accuracy:  89.42%
 - Precision: 81.32%
 - Recall:    89.51%
 - F1-Score:  84.94%


In [47]:
y_train_soft = test_nn.predict(x_train)
y_train_soft.shape

   32/43869 [..............................] - ETA: 15s

43869/43869 [==============================] - 14s 312us/sample


(43869, 10)

In [48]:
x_train_flat = x_train.reshape((x_train.shape[0], -1))
x_valid_flat = x_valid.reshape((x_valid.shape[0], -1))
x_test_flat = x_test.reshape((x_test.shape[0], -1))

# import matplotlib.pyplot as plt
# %matplotlib inline
# plt.imshow(x_test_flat.reshape((x_test_flat.shape[0], img_rows, img_cols))[1])

x_train_flat.shape, x_valid_flat.shape, x_test_flat.shape

((43869, 784), (10000, 784), (10000, 784))

In [49]:
n_features = img_rows * img_cols * img_chans
tree_depth = 4
penalty_strength = 1e+1
penalty_decay = 0.25
ema_win_size = 1000
inv_temp = 0.01
learning_rate = 5e-03
batch_size = 4

In [50]:
sess = brand_new_tfsession(sess)

test_tree = SoftBinaryDecisionTree(tree_depth, n_features, n_classes,
    penalty_strength=penalty_strength, penalty_decay=penalty_decay,
    inv_temp=inv_temp, ema_win_size=ema_win_size, learning_rate=learning_rate)
test_tree.build_model()

Built tree has 16 leaves out of 31 nodes


In [ ]:
import glob
epochs = 40

import time
start_time = time.time()
print("🟢 학습 시작:", time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(start_time)))

es = EarlyStopping(monitor='val_acc', patience=20, verbose=1)

# '''If you wish to train your own model instead of loading one from checkpoint, remove the checkpoint.'''
# os.remove('assets/distilled/checkpoint')
for f in glob.glob('assets/test-mnist-distill-tree/tree-model*'):
    os.remove(f)

test_tree.maybe_train(
    sess=sess, data_train=(x_train_flat, y_train_soft), data_valid=(x_valid_flat, y_valid),
    batch_size=batch_size, epochs=epochs, callbacks=[es], distill=True, model_path = 'assets/mnist_student/test-mnist-distill-tree/tree-model')

# 4. 종료 시간 기록
end_time = time.time()
print("🔴 학습 종료:", time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(end_time)))

# 5. 총 소요 시간
elapsed = end_time - start_time
print(f"⏱️ 총 학습 시간: {elapsed:.2f}초 ({elapsed / 60:.2f}분)")

🟢 학습 시작: 2025-05-27 22:56:54
Loading trained model from assets/test-mnist-distill-tree/tree-model.
assets/test-mnist-distill-tree/tree-model is not a valid checkpoint. Training from scratch.
Train on 43869 samples, validate on 10000 samples
Epoch 1/40
43869/43869 [==============================] - 53s 1ms/sample - loss: 15.0486 - acc: 0.3921 - val_loss: 16.1864 - val_acc: 0.6617
Epoch 2/40
43869/43869 [==============================] - 54s 1ms/sample - loss: 16.3562 - acc: 0.7047 - val_loss: 16.2826 - val_acc: 0.7590
Epoch 3/40
43869/43869 [==============================] - 50s 1ms/sample - loss: 16.3836 - acc: 0.7699 - val_loss: 16.3234 - val_acc: 0.7906
Epoch 4/40
43869/43869 [==============================] - 42s 956us/sample - loss: 16.4842 - acc: 0.7981 - val_loss: 16.4965 - val_acc: 0.8116
Epoch 5/40
43869/43869 [==============================] - 43s 970us/sample - loss: 16.8252 - acc: 0.8195 - val_loss: 16.8305 - val_acc: 0.8291
Epoch 6/40
43869/43869 [==========================

In [13]:
x_train_flat = x_train.reshape(x_train.shape[0], -1)
# 테스트 데이터도 동일하게 처리
x_test_flat = x_test.reshape(x_test.shape[0], -1)

target_class = 3

from sklearn.metrics import precision_score, recall_score, f1_score, classification_report
import numpy as np

# 전체 테스트셋 평가
feed_dict = {test_tree.input_layer: x_test_flat}
predictions = sess.run(test_tree.output_layer, feed_dict=feed_dict)
pred_classes = np.argmax(predictions, axis=1)
true_classes = np.argmax(y_test, axis=1)

accuracy = np.mean(pred_classes == true_classes)
print(f"\nTest accuracy after unlearning: {accuracy:.4f}")

# 전체 성능 지표
print("\n=== Overall Performance ===")
print(f"Precision (macro): {precision_score(true_classes, pred_classes, average='macro'):.4f}")
print(f"Recall    (macro): {recall_score(true_classes, pred_classes, average='macro'):.4f}")
print(f"F1-score  (macro): {f1_score(true_classes, pred_classes, average='macro'):.4f}")
print("\n=== Full Classification Report ===")

# 🎯 타겟 클래스에 대한 성능
target_class = 3
target_mask = np.argmax(y_test, axis=1) == target_class
if np.sum(target_mask) > 0:
    print("\n=== Target Class Performance ===")
    target_acc = np.mean(pred_classes[target_mask] == true_classes[target_mask])
    print(f"Target class accuracy: {target_acc:.4f}")
    print(f"Target class precision: {precision_score(true_classes[target_mask], pred_classes[target_mask], average='macro', zero_division=0):.4f}")
    print(f"Target class recall:    {recall_score(true_classes[target_mask], pred_classes[target_mask], average='macro', zero_division=0):.4f}")
    print(f"Target class F1-score:  {f1_score(true_classes[target_mask], pred_classes[target_mask], average='macro', zero_division=0):.4f}")

# ✅ 비타겟 클래스 성능
non_target_mask = np.argmax(y_test, axis=1) != target_class
if np.sum(non_target_mask) > 0:
    print("\n=== Non-Target Class Performance ===")
    non_target_acc = np.mean(pred_classes[non_target_mask] == true_classes[non_target_mask])
    print(f"Non-target class accuracy: {non_target_acc:.4f}")
    print(f"Non-target precision: {precision_score(true_classes[non_target_mask], pred_classes[non_target_mask], average='macro', zero_division=0):.4f}")
    print(f"Non-target recall:    {recall_score(true_classes[non_target_mask], pred_classes[non_target_mask], average='macro', zero_division=0):.4f}")
    print(f"Non-target F1-score:  {f1_score(true_classes[non_target_mask], pred_classes[non_target_mask], average='macro', zero_division=0):.4f}")



Test accuracy after unlearning: 0.8062

=== Overall Performance ===
Precision (macro): 0.7322
Recall    (macro): 0.8047
F1-score  (macro): 0.7649

=== Full Classification Report ===

=== Target Class Performance ===
Target class accuracy: 0.0000
Target class precision: 0.0000
Target class recall:    0.0000
Target class F1-score:  0.0000

=== Non-Target Class Performance ===
Non-target class accuracy: 0.8968
Non-target precision: 0.8942
Non-target recall:    0.8941
Non-target F1-score:  0.8937


c:\Users\kimdd\anaconda3\envs\unlearning\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


### CiFAR-10

In [1]:
import numpy as np
import tensorflow as tf

from models.convnet_cifar import ConvNet_CIFAR
from models.cifar_tree import SoftBinaryDecisionTree
from models.utils import brand_new_tfsession, draw_tree
from tensorflow.keras.callbacks import EarlyStopping, Callback, ReduceLROnPlateau, LearningRateScheduler
from tensorflow.keras.applications import ResNet50

# 새 TF 세션 생성
sess = brand_new_tfsession()

c:\Users\kimdd\anaconda3\envs\unlearning\lib\site-packages\tensorflow\python\framework\dtypes.py:526: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
c:\Users\kimdd\anaconda3\envs\unlearning\lib\site-packages\tensorflow\python\framework\dtypes.py:527: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_quint8 = np.dtype([("quint8", np.uint8, 1)])
c:\Users\kimdd\anaconda3\envs\unlearning\lib\site-packages\tensorflow\python\framework\dtypes.py:528: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint16 = np.dtype([("qint16", np.int16, 1)])
c:\Users\kimdd\anaconda3\envs\unlearning\lib\site-packages

In [2]:
import tensorflow as tf
import numpy as np

# CIFAR-10 데이터 로드
cifar10 = tf.keras.datasets.cifar10
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# ❗️Remove target class 0 from training data only
train_mask = y_train.flatten() != 0
x_train = x_train[train_mask]
y_train = y_train[train_mask]

# ✅ 검증 및 테스트셋은 class 0 포함한 원본 그대로 사용
x_valid, y_valid = x_train[-10000:], y_train[-10000:]
x_train, y_train = x_train[:-10000], y_train[:-10000]

print(x_train.shape, y_train.shape, x_valid.shape, y_valid.shape, x_test.shape, y_test.shape)

(35000, 32, 32, 3) (35000, 1) (10000, 32, 32, 3) (10000, 1) (10000, 32, 32, 3) (10000, 1)


In [3]:
# 이미지 및 클래스 정보 추출
img_rows, img_cols, img_chans = x_train.shape[1:]
# n_classes = np.unique(y_train).shape[0]
n_classes = 10
print(img_rows, img_cols, img_chans, n_classes)

32 32 3 10


In [4]:
# Convert pixel values data type to float32
x_train = x_train.astype('float32')
x_test  = x_test.astype('float32')
x_valid = x_valid.astype('float32')

# Calculate the mean and standard deviation of the training images
mean = np.mean(x_train, axis = (0, 1, 2, 3))
std  = np.std(x_train, axis = (0, 1, 2, 3))

# 입력 정규화 및 float32 캐스팅
x_train = (x_train-mean)/(std+1e-7)
x_test  = (x_test-mean) /(std+1e-7)
x_valid = (x_valid-mean)/(std+1e-7)

In [5]:
# label을 1-hot 벡터로 변환
y_train = tf.keras.utils.to_categorical(y_train, n_classes)
y_valid = tf.keras.utils.to_categorical(y_valid, n_classes)
y_test = tf.keras.utils.to_categorical(y_test, n_classes)

print(y_train.shape, y_valid.shape, y_test.shape)

(35000, 10) (10000, 10) (10000, 10)


In [6]:
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=10, min_lr=1e-06, verbose = 1)

early_stopping = EarlyStopping(monitor='val_loss', patience=40, restore_best_weights=True, verbose=1)

In [ ]:
# CNN 모델 정의 및 학습
import time
start_time = time.time()
print("🟢 학습 시작:", time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(start_time)))

cifar_test = ConvNet_CIFAR(img_rows, img_cols, img_chans, n_classes, model_path = 'assets/cifar-10_teacher/cifar10-test2.hdf5')
cifar_test.maybe_train(data_train=(x_train, y_train),
               data_valid=(x_valid, y_valid),
               batch_size=64, callbacks = [early_stopping, reduce_lr], epochs=50)  # CIFAR-10은 조금 더 epoch 필요

# 4. 종료 시간 기록
end_time = time.time()
print("🔴 학습 종료:", time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(end_time)))

# 5. 총 소요 시간
elapsed = end_time - start_time
print(f"⏱️ 총 학습 시간: {elapsed:.2f}초 ({elapsed / 60:.2f}분)")

🟢 학습 시작: 2025-05-31 10:43:12
Instructions for updating:
Colocations handled automatically by placer.
🚀 No checkpoint found on assets/cifar10-test2.hdf5. Training from scratch.
Instructions for updating:
Please use `rate` instead of `keep_prob`. Rate should be set to `rate = 1 - keep_prob`.
Train on 35000 samples, validate on 10000 samples
Instructions for updating:
Use tf.cast instead.
Epoch 1/50
35000/35000 [==============================] - 94s 3ms/sample - loss: 1.7282 - acc: 0.3829 - val_loss: 1.3216 - val_acc: 0.5128
Epoch 2/50
35000/35000 [==============================] - 93s 3ms/sample - loss: 1.3122 - acc: 0.5359 - val_loss: 1.0207 - val_acc: 0.6348
Epoch 3/50
35000/35000 [==============================] - 92s 3ms/sample - loss: 1.1086 - acc: 0.6200 - val_loss: 0.9157 - val_acc: 0.6848
Epoch 4/50
35000/35000 [==============================] - 93s 3ms/sample - loss: 0.9814 - acc: 0.6749 - val_loss: 0.8771 - val_acc: 0.7097
Epoch 5/50
35000/35000 [==============================]

In [36]:
# Soft label 추출
y_train_soft = cifar_test.predict(x_train, batch_size = 64)
print(y_train_soft.shape)


# def sharpen_soft_labels(soft_labels, T=0.5, eps=1e-8):
#     soft_labels = np.clip(soft_labels, eps, 1.0)  # 0 방지
#     soft_labels = np.power(soft_labels, 1.0 / T)
#     soft_labels = soft_labels / np.sum(soft_labels, axis=1, keepdims=True)
#     return soft_labels

# y_train_soft = sharpen_soft_labels(y_train_soft, T = 0.5)

35000/35000 [==============================] - 15s 417us/sample
(35000, 10)


In [30]:
for layer in cifar_test.model.layers:
    print(layer.name)

input_1
conv2d
activation
batch_normalization_v1
conv2d_1
activation_1
batch_normalization_v1_1
max_pooling2d
dropout
conv2d_2
activation_2
batch_normalization_v1_2
conv2d_3
activation_3
batch_normalization_v1_3
max_pooling2d_1
dropout_1
conv2d_4
activation_4
batch_normalization_v1_4
conv2d_5
activation_5
batch_normalization_v1_5
max_pooling2d_2
dropout_2
flatten
dense
dropout_3
dense_1


In [37]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D
from sklearn.decomposition import PCA
import numpy as np

# (1) Feature 추출 모델 정의 (conv2d_5 → max_pooling2d_2 → GAP)
intermediate_layer = cifar_test.model.get_layer('max_pooling2d_2').output # max_pooling2d_2
gap_output = GlobalAveragePooling2D()(intermediate_layer)

feature_model = Model(
    inputs=cifar_test.model.input,
    outputs=gap_output
)

# (2) Feature 추출
x_train_flat = feature_model.predict(x_train, batch_size=64)
x_valid_flat = feature_model.predict(x_valid, batch_size=64)
x_test_flat  = feature_model.predict(x_test,  batch_size=64)

print(x_train_flat.shape, x_valid_flat.shape, x_test_flat.shape)

(35000, 128) (10000, 128) (10000, 128)


In [38]:
# 0.8385
n_features = x_train_flat.shape[1]

sess = brand_new_tfsession()

cifar_test_tree = SoftBinaryDecisionTree(
    max_depth=5,
    n_features=n_features,
    n_classes=n_classes,
    penalty_strength= 0.40895213388013624,
    penalty_decay=0.49627977113163907,
    ema_win_size=1000,
    inv_temp=0.038985814991530615,
    learning_rate=0.00209034548768749
)
cifar_test_tree.build_model()

Built tree has 32 leaves out of 63 nodes


In [ ]:
# import glob
# for f in glob.glob('assets/cifar10-test-distill-tree/tree-model*'):
#     os.remove(f)

import time
start_time = time.time()
print("🟢 학습 시작:", time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(start_time)))

cifar_test_tree.initialize_variables(sess, x_train_flat, batch_size = 4)

cifar_test_tree.maybe_train(
    sess=sess,
    data_train=(x_train_flat, y_train_soft),
    data_valid=(x_valid_flat, y_valid),
    batch_size=4,
    epochs=40,
    callbacks=[EarlyStopping(monitor='val_acc', patience=10)],
    distill=True,
    model_path="assets/cifar-10_student/cifar10-test-distill-tree/tree-model"
)

# 4. 종료 시간 기록
end_time = time.time()
print("🔴 학습 종료:", time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(end_time)))

# 5. 총 소요 시간
elapsed = end_time - start_time
print(f"⏱️ 총 학습 시간: {elapsed:.2f}초 ({elapsed / 60:.2f}분)")

🟢 학습 시작: 2025-05-27 22:05:02
Loading trained model from assets/cifar10-test-distill-tree/tree-model.
assets/cifar10-test-distill-tree/tree-model is not a valid checkpoint. Training from scratch.
Train on 35000 samples, validate on 10000 samples
Epoch 1/40
35000/35000 [==============================] - 75s 2ms/sample - loss: 1.9996 - acc: 0.3953 - val_loss: 1.9186 - val_acc: 0.5441
Epoch 2/40
35000/35000 [==============================] - 69s 2ms/sample - loss: 1.9200 - acc: 0.6079 - val_loss: 1.9252 - val_acc: 0.6122
Epoch 3/40
35000/35000 [==============================] - 68s 2ms/sample - loss: 1.9284 - acc: 0.6935 - val_loss: 1.9397 - val_acc: 0.7102
Epoch 4/40
35000/35000 [==============================] - 69s 2ms/sample - loss: 1.9408 - acc: 0.7637 - val_loss: 1.9478 - val_acc: 0.7365
Epoch 5/40
35000/35000 [==============================] - 69s 2ms/sample - loss: 1.9474 - acc: 0.7896 - val_loss: 1.9539 - val_acc: 0.7555
Epoch 6/40
35000/35000 [==============================] - 83

In [12]:
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

# 🎯 타겟 클래스 정의
target_class = 0

# ✅ 예측 수행
feed_dict = {cifar_test_tree.input_layer: x_test_flat}
predictions = sess.run(cifar_test_tree.output_layer, feed_dict=feed_dict)

# ✅ 예측 및 실제 클래스
pred_classes = np.argmax(predictions, axis=1)
true_classes = np.argmax(y_test, axis=1)

# ✅ 전체 정확도 및 전체 성능 지표
print(f"\n📊 Test accuracy after unlearning: {np.mean(pred_classes == true_classes):.4f}")

print("\n=== Overall Performance ===")
print(f"Precision (macro): {precision_score(true_classes, pred_classes, average='macro'):.4f}")
print(f"Recall    (macro): {recall_score(true_classes, pred_classes, average='macro'):.4f}")
print(f"F1-score  (macro): {f1_score(true_classes, pred_classes, average='macro'):.4f}")

# ✅ 마스크 생성 (target / non-target)
target_mask = true_classes == target_class
non_target_mask = ~target_mask

# 🎯 타겟 클래스 성능
if np.any(target_mask):
    print("\n=== Target Class Performance ===")
    print(f"Accuracy:  {np.mean(pred_classes[target_mask] == true_classes[target_mask]):.4f}")
    print(f"Precision: {precision_score(true_classes[target_mask], pred_classes[target_mask], average='macro', zero_division=0):.4f}")
    print(f"Recall:    {recall_score(true_classes[target_mask], pred_classes[target_mask], average='macro', zero_division=0):.4f}")
    print(f"F1-score:  {f1_score(true_classes[target_mask], pred_classes[target_mask], average='macro', zero_division=0):.4f}")

# ✅ 비타겟 클래스 성능
if np.any(non_target_mask):
    print("\n=== Non-Target Class Performance ===")
    print(f"Accuracy:  {np.mean(pred_classes[non_target_mask] == true_classes[non_target_mask]):.4f}")
    print(f"Precision: {precision_score(true_classes[non_target_mask], pred_classes[non_target_mask], average='macro', zero_division=0):.4f}")
    print(f"Recall:    {recall_score(true_classes[non_target_mask], pred_classes[non_target_mask], average='macro', zero_division=0):.4f}")
    print(f"F1-score:  {f1_score(true_classes[non_target_mask], pred_classes[non_target_mask], average='macro', zero_division=0):.4f}")



📊 Test accuracy after unlearning: 0.7151

=== Overall Performance ===
Precision (macro): 0.6498
Recall    (macro): 0.7151
F1-score  (macro): 0.6787

=== Target Class Performance ===
Accuracy:  0.0000
Precision: 0.0000
Recall:    0.0000
F1-score:  0.0000

=== Non-Target Class Performance ===
Accuracy:  0.7946
Precision: 0.7950
Recall:    0.7946
F1-score:  0.7946


c:\Users\kimdd\anaconda3\envs\unlearning\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


## FashionMNIST

In [7]:
import numpy as np
import os
import tensorflow as tf

from models import ConvNet, SoftBinaryDecisionTree
from models.unlearning import UnlearningSDT
from models.utils import brand_new_tfsession, draw_tree
from tensorflow.keras.callbacks import EarlyStopping, Callback

# 새 세션 시작
sess = brand_new_tfsession()

In [8]:
# ✅ FashionMNIST 데이터 로드
fashion_mnist = tf.keras.datasets.fashion_mnist
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

x_train = x_train[..., np.newaxis]
x_test = x_test[..., np.newaxis]

# ❗️Remove samples with target class 3 from training data only
train_mask = y_train != 0
x_train = x_train[train_mask]
y_train = y_train[train_mask]

# 마지막 10000개를 검증용으로 분리
x_valid, y_valid = x_train[-10000:], y_train[-10000:]
x_train, y_train = x_train[:-10000], y_train[:-10000]

print(x_train.shape, y_train.shape, x_valid.shape, y_valid.shape, x_test.shape, y_test.shape)


(44000, 28, 28, 1) (44000,) (10000, 28, 28, 1) (10000,) (10000, 28, 28, 1) (10000,)


In [9]:
# 이미지 크기 및 클래스 수 추출
img_rows, img_cols, img_chans = x_train.shape[1:]
# n_classes = np.unique(y_train).shape[0]
n_classes = 10
print(img_rows, img_cols, img_chans, n_classes)

28 28 1 10


In [10]:
# 원-핫 인코딩
y_train = tf.keras.utils.to_categorical(y_train, n_classes)
y_valid = tf.keras.utils.to_categorical(y_valid, n_classes)
y_test = tf.keras.utils.to_categorical(y_test, n_classes)
print(y_train.shape, y_valid.shape, y_test.shape)

(44000, 10) (10000, 10) (10000, 10)


In [11]:
# 정규화
x_train = (x_train / 255.0).astype(np.float32)
x_valid = (x_valid / 255.0).astype(np.float32)
x_test = (x_test / 255.0).astype(np.float32)


In [ ]:
# ConvNet 학습
import time
start_time = time.time()
print("🟢 학습 시작:", time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(start_time)))

Fashion_test = ConvNet(img_rows, img_cols, img_chans, n_classes,
             model_path='assets/fashionmnist_teacher/fashionmnist-test2-model.hdf5')  # <- 별도 저장 권장
Fashion_test.maybe_train(data_train=(x_train, y_train),
               data_valid=(x_valid, y_valid),
               batch_size=16, epochs=20)  

# 4. 종료 시간 기록
end_time = time.time()
print("🔴 학습 종료:", time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(end_time)))

# 5. 총 소요 시간
elapsed = end_time - start_time
print(f"⏱️ 총 학습 시간: {elapsed:.2f}초 ({elapsed / 60:.2f}분)")

🟢 학습 시작: 2025-05-29 12:42:07
🚀 No checkpoint found on assets/fashionmnist-test2-model.hdf5. Training from scratch.
Train on 44000 samples, validate on 10000 samples
Epoch 1/20
44000/44000 [==============================] - 57s 1ms/sample - loss: 0.4501 - acc: 0.8344 - val_loss: 0.2690 - val_acc: 0.9009
Epoch 2/20
44000/44000 [==============================] - 56s 1ms/sample - loss: 0.2724 - acc: 0.9026 - val_loss: 0.2255 - val_acc: 0.9176
Epoch 3/20
44000/44000 [==============================] - 57s 1ms/sample - loss: 0.2210 - acc: 0.9200 - val_loss: 0.1889 - val_acc: 0.9307
Epoch 4/20
44000/44000 [==============================] - 56s 1ms/sample - loss: 0.1934 - acc: 0.9296 - val_loss: 0.1753 - val_acc: 0.9365
Epoch 5/20
44000/44000 [==============================] - 57s 1ms/sample - loss: 0.1741 - acc: 0.9359 - val_loss: 0.1665 - val_acc: 0.9404
Epoch 6/20
44000/44000 [==============================] - 58s 1ms/sample - loss: 0.1583 - acc: 0.9421 - val_loss: 0.1893 - val_acc: 0.9353
E

In [14]:
# Soft label 생성
y_train_soft = Fashion_test.predict(x_train)

44000/44000 [==============================] - 14s 308us/sample


In [15]:
# Flatten for SDT
x_train_flat = x_train.reshape((x_train.shape[0], -1))
x_valid_flat = x_valid.reshape((x_valid.shape[0], -1))
x_test_flat = x_test.reshape((x_test.shape[0], -1))

print(x_train_flat.shape, x_valid_flat.shape, x_test_flat.shape)

(44000, 784) (10000, 784) (10000, 784)


In [16]:
# 0.8518
n_features = img_rows * img_cols * img_chans
sess = brand_new_tfsession()  # 세션 초기화
fashion_test_tree = SoftBinaryDecisionTree(
    max_depth=5,
    n_features=n_features,
    n_classes=n_classes,
    penalty_strength=0.4048333255228549,
    penalty_decay=0.03336682424444432,
    ema_win_size=900,
    inv_temp=0.059875870091768986,
    learning_rate=0.003266302354704365
)
fashion_test_tree.build_model()

Built tree has 32 leaves out of 63 nodes


In [ ]:
import glob
import os
# for f in glob.glob('assets/fashion_tree-test-distill-tree/tree-model*'):
#     os.remove(f)

import time
start_time = time.time()
print("🟢 학습 시작:", time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(start_time)))

fashion_test_tree.maybe_train(
    sess=sess,
    data_train=(x_train_flat, y_train_soft),
    data_valid=(x_valid_flat, y_valid),
    batch_size=4,
    epochs=40,
    callbacks=[EarlyStopping(monitor='val_acc', patience=10)],
    distill=True,
    model_path="assets/fashionmnist_student/fashion_tree-test-distill-tree/tree-model"
)

# 4. 종료 시간 기록
end_time = time.time()
print("🔴 학습 종료:", time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(end_time)))

# 5. 총 소요 시간
elapsed = end_time - start_time
print(f"⏱️ 총 학습 시간: {elapsed:.2f}초 ({elapsed / 60:.2f}분)")

🟢 학습 시작: 2025-05-28 20:27:26
Loading trained model from assets/fashion_tree-test-distill-tree/tree-model.
Instructions for updating:
Use standard file APIs to check for files with this prefix.
assets/fashion_tree-test-distill-tree/tree-model is not a valid checkpoint. Training from scratch.
Train on 44000 samples, validate on 10000 samples
Instructions for updating:
Use tf.cast instead.
Epoch 1/40
44000/44000 [==============================] - 77s 2ms/sample - loss: 0.8282 - acc: 0.3306 - val_loss: 0.8132 - val_acc: 0.4797
Epoch 2/40
44000/44000 [==============================] - 78s 2ms/sample - loss: 0.8127 - acc: 0.4946 - val_loss: 0.8127 - val_acc: 0.4895
Epoch 3/40
44000/44000 [==============================] - 72s 2ms/sample - loss: 0.8132 - acc: 0.5048 - val_loss: 0.8143 - val_acc: 0.5052
Epoch 4/40
44000/44000 [==============================] - 70s 2ms/sample - loss: 0.8123 - acc: 0.5104 - val_loss: 0.8145 - val_acc: 0.5047
Epoch 5/40
44000/44000 [==============================

In [12]:
import numpy as np
from sklearn.metrics import accuracy_score

# 🔹 전체 테스트셋 예측
predictions = fashion_test_tree.model.predict(x_test_flat, batch_size=4)
pred_classes = np.argmax(predictions, axis=1)
true_classes = np.argmax(y_test, axis=1)

accuracy = np.mean(pred_classes == true_classes)

# 🔹 전체 정확도
overall_acc = accuracy_score(true_classes, pred_classes)
print(f"✅ Overall accuracy: {overall_acc:.4f}")

# 🔹 타겟 클래스 (0) 정확도
target_class = 0
target_mask = true_classes == target_class
if np.sum(target_mask) > 0:
    target_acc = accuracy_score(true_classes[target_mask], pred_classes[target_mask])
    print(f"🎯 Target class ({target_class}) accuracy: {target_acc:.4f}")

# 🔹 비타겟 클래스 정확도
non_target_mask = true_classes != target_class
non_target_acc = accuracy_score(true_classes[non_target_mask], pred_classes[non_target_mask])
print(f"📦 Non-target class accuracy: {non_target_acc:.4f}")


✅ Overall accuracy: 0.6857
🎯 Target class (0) accuracy: 0.0000
📦 Non-target class accuracy: 0.7619


### SVHN

In [1]:
import numpy as np
import tensorflow as tf

from models.convnet_cifar import ConvNet_CIFAR
# from models.convnet_svhn import ConvNet_SVHN
from models.svhn_tree import SoftBinaryDecisionTree
from models.utils import brand_new_tfsession, draw_tree
from tensorflow.keras.callbacks import EarlyStopping, Callback, ReduceLROnPlateau, LearningRateScheduler
from tensorflow.keras.applications import ResNet50

import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
from sklearn.model_selection import train_test_split
# 새 TF 세션 생성
sess = brand_new_tfsession()

c:\Users\kimdd\anaconda3\envs\unlearning\lib\site-packages\tensorflow\python\framework\dtypes.py:526: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
c:\Users\kimdd\anaconda3\envs\unlearning\lib\site-packages\tensorflow\python\framework\dtypes.py:527: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_quint8 = np.dtype([("quint8", np.uint8, 1)])
c:\Users\kimdd\anaconda3\envs\unlearning\lib\site-packages\tensorflow\python\framework\dtypes.py:528: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint16 = np.dtype([("qint16", np.int16, 1)])
c:\Users\kimdd\anaconda3\envs\unlearning\lib\site-packages

In [ ]:
def load_data(path):
    """ Load SVHN .mat file and transpose """
    data = loadmat(path)
    X = data['X'].transpose(3, 0, 1, 2)  # (32,32,3,N) → (N,32,32,3)
    y = data['y'].flatten()
    y[y == 10] = 0
    return X, y

x_train_full, y_train_full = load_data("./data/train_32x32.mat")
x_test, y_test = load_data("./data/test_32x32.mat")

In [3]:
print("원본 y_train_full 레이블 분포:", np.unique(y_train_full))

원본 y_train_full 레이블 분포: [0 1 2 3 4 5 6 7 8 9]


In [4]:
# 2. Split into train / valid
train_mask = y_train_full != 0
x_train_filtered = x_train_full[train_mask]
y_train_filtered = y_train_full[train_mask]

# ✅ 2. filtered dataset에서 train/valid split
x_train, x_valid, y_train, y_valid = train_test_split(
    x_train_filtered, y_train_filtered,
    test_size=0.3, random_state=42, stratify=y_train_filtered
)


# 3. 정확한 개수로 자르기
# x_train, y_train = x_train[:58600], y_train[:58600]
# x_valid, y_valid = x_valid[:14650], y_valid[:14650]
# x_test, y_test   = x_test[:26030], y_test[:26030]

In [5]:
print("Train shape:", x_train.shape, y_train.shape)
print("Valid shape:", x_valid.shape, y_valid.shape)
print("Test shape :", x_test.shape, y_test.shape)

Train shape: (47816, 32, 32, 3) (47816,)
Valid shape: (20493, 32, 32, 3) (20493,)
Test shape : (26032, 32, 32, 3) (26032,)


In [6]:
# 이미지 및 클래스 정보 추출
img_rows, img_cols, img_chans = x_train.shape[1:]
# n_classes = np.unique(y_train).shape[0]
n_classes = 10
print(img_rows, img_cols, img_chans, n_classes)

32 32 3 10


In [7]:
# float32 변환
x_train = x_train.astype('float32')
x_valid = x_valid.astype('float32')
x_test  = x_test.astype('float32')

# mean/std는 train 기준으로만
mean = np.mean(x_train, axis=(0, 1, 2, 3))
std  = np.std(x_train, axis=(0, 1, 2, 3))

x_train = (x_train - mean) / (std + 1e-7)
x_valid = (x_valid - mean) / (std + 1e-7)
x_test  = (x_test - mean) / (std + 1e-7)


In [8]:
# y_train[y_train == 10] = 0
# y_valid[y_valid == 10] = 0
# y_test[y_test == 10] = 0

# 3. one-hot 인코딩
y_train = tf.keras.utils.to_categorical(y_train, num_classes=10)
y_valid = tf.keras.utils.to_categorical(y_valid, num_classes=10)
y_test  = tf.keras.utils.to_categorical(y_test, num_classes=10)

In [9]:
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=10, min_lr=1e-05, verbose = 1)

early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)

In [ ]:
# CNN 모델 정의 및 학습

svhn_test = ConvNet_CIFAR(img_rows, img_cols, img_chans, n_classes, model_path = 'assets/svhn_teacher/svhn-model-test.hdf5')
svhn_test.maybe_train(data_train=(x_train, y_train),
               data_valid=(x_valid, y_valid),
               batch_size=64, callbacks = [early_stopping, reduce_lr], epochs=50)  # CIFAR-10은 조금 더 epoch 필요

Instructions for updating:
Colocations handled automatically by placer.
🔁 Trying to load full model from assets/svhn-model-test.hdf5.
Instructions for updating:
Please use `rate` instead of `keep_prob`. Rate should be set to `rate = 1 - keep_prob`.


In [11]:
# Soft label 추출
y_train_soft = svhn_test.predict(x_train, batch_size = 64)
print(y_train_soft.shape)


# def sharpen_soft_labels(soft_labels, T=0.5, eps=1e-8):
#     soft_labels = np.clip(soft_labels, eps, 1.0)  # 0 방지
#     soft_labels = np.power(soft_labels, 1.0 / T)
#     soft_labels = soft_labels / np.sum(soft_labels, axis=1, keepdims=True)
#     return soft_labels

# y_train_soft = sharpen_soft_labels(y_train_soft, T = 0.5)

47816/47816 [==============================] - 20s 412us/sample
(47816, 10)


In [12]:
for layer in svhn_test.model.layers:
    print(layer.name)

input_1
conv2d
activation
batch_normalization_v1
conv2d_1
activation_1
batch_normalization_v1_1
max_pooling2d
dropout
conv2d_2
activation_2
batch_normalization_v1_2
conv2d_3
activation_3
batch_normalization_v1_3
max_pooling2d_1
dropout_1
conv2d_4
activation_4
batch_normalization_v1_4
conv2d_5
activation_5
batch_normalization_v1_5
max_pooling2d_2
dropout_2
flatten
dense
dropout_3
dense_1


In [14]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D
from sklearn.decomposition import PCA
import numpy as np

# (1) Feature 추출 모델 정의 (conv2d_5 → max_pooling2d_2 → GAP)
intermediate_layer = svhn_test.model.get_layer('max_pooling2d_2').output # max_pooling2d_2
gap_output = GlobalAveragePooling2D()(intermediate_layer)

feature_model = Model(
    inputs=svhn_test.model.input,
    outputs=gap_output
)

# (2) Feature 추출
x_train_flat = feature_model.predict(x_train, batch_size=64)
x_valid_flat = feature_model.predict(x_valid, batch_size=64)
x_test_flat  = feature_model.predict(x_test,  batch_size=64)

print(x_train_flat.shape, x_valid_flat.shape, x_test_flat.shape)

(47816, 128) (20493, 128) (26032, 128)


In [ ]:
import optuna
from tensorflow.keras.callbacks import EarlyStopping
import glob
import os

# n_features = img_rows * img_cols * img_chans
n_features = x_train_flat.shape[1]
# batch_size = 4

# def sharpen_soft_labels(soft_labels, T):
#     softened = np.power(soft_labels, 1.0 / T)
#     softened = softened / np.sum(softened, axis=1, keepdims=True)
#     return softened

def objective(trial):
    sess = brand_new_tfsession()
    # 1. 튜닝할 하이퍼파라미터 설정
    tree_depth = trial.suggest_int("tree_depth", 5, 5)
    penalty_strength = trial.suggest_loguniform("penalty_strength", 0.01, 5.0)
    penalty_decay = trial.suggest_uniform("penalty_decay", 0.01, 0.5)
    ema_win_size = trial.suggest_int("ema_win_size", 500, 2000, step = 100)
    inv_temp = trial.suggest_uniform("inv_temp", 0.01, 1.0)
    learning_rate = trial.suggest_float("learning_rate", 5e-04, 1e-02, log = True)
    batch_size = trial.suggest_categorical("batch_size", [4, 8, 16])
    # tree_depth = trial.suggest_int("tree_depth", 4, 5)
    # penalty_strength = trial.suggest_loguniform("penalty_strength", 1e-2, 1e+1)
    # penalty_decay = trial.suggest_uniform("penalty_decay", 0.01, 0.5)
    # ema_win_size = trial.suggest_int("ema_win_size", 500, 1000, step = 100)
    # inv_temp = trial.suggest_uniform("inv_temp", 0.01, 0.1)
    # learning_rate = trial.suggest_float("learning_rate", 5e-04, 1e-02, log = True)
    # batch_size = trial.suggest_categorical("batch_size", [4, 8, 16])
    # T = trial.suggest_float("sharpen_T", 0.3, 1.0)  # 일반적으로 0.3 ~ 1.0
    # softened_labels = sharpen_soft_labels(y_train_soft, T=T)
    
    # 고유한 체크포인트 경로 생성 (Trial 번호 기반)
    checkpoint_dir = f"assets/svhn_optuna_test_trial/trial_{trial.number}/"
    checkpoint_path = os.path.join(checkpoint_dir, "tree-model")

    # 💡 기존 체크포인트가 있다면 삭제
    if os.path.isdir(checkpoint_dir):
        old_ckpts = glob.glob(os.path.join(checkpoint_dir, "tree-model*"))
        for file in old_ckpts:
            os.remove(file)
    else:
        os.makedirs(checkpoint_dir, exist_ok=True)


    # 2. 트리 모델 생성
    sess = brand_new_tfsession()  # 세션 초기화
    svhn_tree = SoftBinaryDecisionTree(
        tree_depth, n_features, n_classes,
        penalty_strength=penalty_strength,
        penalty_decay=penalty_decay,
        inv_temp=inv_temp,
        ema_win_size=ema_win_size,
        learning_rate=learning_rate
    )
    svhn_tree.build_model()

    # 3. 학습
    es = EarlyStopping(monitor='val_acc', patience=5, verbose=0)
    rl = ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=5, min_lr=1e-06, verbose = 1)
    svhn_tree.initialize_variables(sess, x_train_flat, batch_size = batch_size)

    svhn_tree.maybe_train(
        sess = sess,
        data_train = (x_train_flat, y_train_soft),
        data_valid=(x_valid_flat, y_valid),
        batch_size=batch_size,
        epochs=30,  # 빠른 튜닝용 짧은 epoch (본 학습 때는 늘리면 됨)
        callbacks=[es, rl],
        model_path=checkpoint_path
    )

    # 4. 평가 지표 반환 (validation accuracy)
    score = svhn_tree.cifar_evaluate(x_valid_flat, y_valid, batch_size)
    tf.reset_default_graph()
    return score[1]  # val_acc

# 5. Optuna 튜닝 시작
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)  # 20번 실험 (조절 가능)

# 6. 최적 파라미터 확인
print("🎯 Best trial:")
print(study.best_trial)


In [18]:
sess = brand_new_tfsession()

n_features = x_train_flat.shape[1]

svhn_test_tree = SoftBinaryDecisionTree(
    max_depth=5,
    n_features=n_features,
    n_classes=n_classes,
    penalty_strength= 4.96750975303067,
    penalty_decay=0.3531287598419813,
    ema_win_size=900,
    inv_temp=0.16467147809062108,
    learning_rate=0.0032629572226774003
)
svhn_test_tree.build_model()

Built tree has 32 leaves out of 63 nodes


In [ ]:
import glob
import os
# for f in glob.glob('assets/svhn-test-distill-tree/tree-model*'):
#     os.remove(f)

svhn_test_tree.initialize_variables(sess, x_train_flat, batch_size = 4)

svhn_test_tree.maybe_train(
    sess=sess,
    data_train=(x_train_flat, y_train_soft),
    data_valid=(x_valid_flat, y_valid),
    batch_size=4,
    epochs=40,
    callbacks=[EarlyStopping(monitor='val_acc', patience=10)],
    distill=True,
    model_path="assets/svhn_student/svhn-test-distill-tree/tree-model"
)

⚠️ Checkpoint load failed: Checkpoint not found.
🔁 Training from scratch.
Train on 47816 samples, validate on 20493 samples
Epoch 1/40
47816/47816 [==============================] - 94s 2ms/sample - loss: 0.7516 - acc: 0.2081 - val_loss: 0.7544 - val_acc: 0.2029
Epoch 2/40
47816/47816 [==============================] - 95s 2ms/sample - loss: 0.7447 - acc: 0.2445 - val_loss: 0.7454 - val_acc: 0.2495
Epoch 3/40
47816/47816 [==============================] - 95s 2ms/sample - loss: 0.7390 - acc: 0.2571 - val_loss: 0.7421 - val_acc: 0.2501
Epoch 4/40
47816/47816 [==============================] - 99s 2ms/sample - loss: 0.7377 - acc: 0.2558 - val_loss: 0.7416 - val_acc: 0.2479
Epoch 5/40
47816/47816 [==============================] - 86s 2ms/sample - loss: 0.7368 - acc: 0.2609 - val_loss: 0.7399 - val_acc: 0.2859
Epoch 6/40
47816/47816 [==============================] - 87s 2ms/sample - loss: 0.7341 - acc: 0.3208 - val_loss: 0.7371 - val_acc: 0.3502
Epoch 7/40
47816/47816 [==================

In [20]:
import numpy as np
from sklearn.metrics import accuracy_score

# 🔹 전체 테스트셋 예측
predictions = svhn_test_tree.model.predict(x_test_flat, batch_size=4)
pred_classes = np.argmax(predictions, axis=1)
true_classes = np.argmax(y_test, axis=1)

accuracy = np.mean(pred_classes == true_classes)

# 🔹 전체 정확도
overall_acc = accuracy_score(true_classes, pred_classes)
print(f"✅ Overall accuracy: {overall_acc:.4f}")

# 🔹 타겟 클래스 (0) 정확도
target_class = 0
target_mask = true_classes == target_class
if np.sum(target_mask) > 0:
    target_acc = accuracy_score(true_classes[target_mask], pred_classes[target_mask])
    print(f"🎯 Target class ({target_class}) accuracy: {target_acc:.4f}")

# 🔹 비타겟 클래스 정확도
non_target_mask = true_classes != target_class
non_target_acc = accuracy_score(true_classes[non_target_mask], pred_classes[non_target_mask])
print(f"📦 Non-target class accuracy: {non_target_acc:.4f}")


✅ Overall accuracy: 0.7168
🎯 Target class (0) accuracy: 0.0000
📦 Non-target class accuracy: 0.7683
